In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
os.chdir('../')

In [3]:
from src.end_to_end_deep_learning_project_yt.entity.config_entity import (
    DataIngestionConfig
    , PrepareBaseModelConfig
    , PrepareCallbacksConfig
)

In [4]:
from pydantic import BaseModel, ConfigDict, Field
from pathlib import Path

# Training Params
class Params(BaseModel):
    model_config = ConfigDict(extra='ignore', populate_by_name=True)

    image_size: list[int] = Field(..., alias='IMAGE_SIZE')
    epochs: int = Field(..., alias='EPOCHS')
    batch_size: int = Field(..., alias='BATCH_SIZE')
    is_augmentation: bool = Field(..., alias='AUGMENTATION')

class TrainingBaseConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    root_dir: Path
    trained_model_path: Path

class Config(BaseModel):
    model_config = ConfigDict(extra='forbid')

    artifacts_root: Path
    data_ingestion: DataIngestionConfig
    prepare_base_model: PrepareBaseModelConfig
    prepare_callbacks: PrepareCallbacksConfig
    training: TrainingBaseConfig

class TrainingConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    base_config: TrainingBaseConfig
    updated_base_model_path: Path
    training_data: Path
    params: Params

In [5]:
from src.end_to_end_deep_learning_project_yt.constants import constants
from src.end_to_end_deep_learning_project_yt.utils.common import read_yaml, create_directories
import tensorflow as tf

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = constants.CONFIG_FILE_PATH,
        params_filepath = constants.PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath, Config)
        self.params = read_yaml(params_filepath, Params)

        create_directories([self.config.artifacts_root])

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        prepare_callback_config = self.config.prepare_callbacks

        model_ckpt_dir = os.path.dirname(prepare_callback_config.checkpoint_model_filepath)

        create_directories([
            Path(model_ckpt_dir),
            Path(prepare_callback_config.tensorboard_root_log_dir)
        ])

        return prepare_callback_config

    def get_training_config(self) -> TrainingConfig:
        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir
            , 'Chicken-fecal-images'
        )

        create_directories([
            Path(self.config.training.root_dir)
        ])

        training_config = TrainingConfig(
            base_config = self.config.training
            , updated_base_model_path = self.config.prepare_base_model.updated_base_model_path
            , training_data = training_data
            , params = self.params
        )

        return training_config

In [7]:
from src.end_to_end_deep_learning_project_yt.components.prepare_callbacks import PrepareCallback

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):
        datagenerators_kwargs = dict(
            rescale = 1. / 255,
            validation_split = 0.2
        )

        dataflow_kwargs = dict(
            target_size = self.config.params.image_size[:-1],
            batch_size = self.config.params.batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerators_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data
            , subset='validation'
            , shuffle=False
            , **dataflow_kwargs
        )

        if self.config.params.is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40
                , horizontal_flip=True
                , width_shift_range=0.2
                , height_shift_range=0.2
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data
            , subset='training'
            , shuffle=True
            , **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self, callback_list: list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator
            , epochs=self.config.params.epochs
            , validation_steps=self.validation_steps
            , validation_data=self.valid_generator
            , callbacks=callback_list
        )

        self.save_model(
            path=self.config.base_config.trained_model_path
            , model=self.model
        )

In [10]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(callback_list=callback_list)
except Exception as e:
    raise e

[2026-11-08 08:29:39,497]: INFO - common - yaml file: config/config.yaml loaded successfully
[2026-11-08 08:29:39,499]: INFO - common - yaml file: params.yaml loaded successfully
[2026-11-08 08:29:39,499]: INFO - common - created directory at: artifacts
[2026-11-08 08:29:39,499]: INFO - common - created directory at: artifacts/prepare_callbacks/checkpoint_dir
[2026-11-08 08:29:39,500]: INFO - common - created directory at: artifacts/prepare_callbacks/tensorboard_log_dir
[2026-11-08 08:29:39,501]: INFO - common - created directory at: artifacts/training
Found 78 images belonging to 2 classes.
Found 390 images belonging to 2 classes.
25/25 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.7590 - loss: 235.3273 - val_accuracy: 0.7031 - val_loss: 3.8466
